In [3]:
"""
gee_extract.py
==============
Google Earth Engine — Raw Data Extraction Script
Harvest Date Prediction Pipeline (Implementation Plan v2)

WHAT THIS SCRIPT DOES
----------------------
1. Authenticates with GEE and initialises the project.
2. Ingests a user-supplied CSV of 1,000 sample points (point_id, lat, lon).
3. Exports THREE separate CSVs to Google Drive (one per satellite source):

   a) sentinel2_raw_<year>.csv  — raw S2 bands (B2,B4,B5,B6,B7,B8,B11,B12) +
                                   SCL cloud mask, per point × date.
   b) sentinel1_raw_<year>.csv  — S1 GRD VH and VV backscatter (dB),
                                   per point × date.
   c) static_layers.csv         — SRTM elevation/slope/aspect + ESA WorldCover
                                   land-cover class, per point (one-time extract).

WHAT THIS SCRIPT DOES NOT DO (by design — see §12, Milestone 0)
-----------------------------------------------------------------
- No temporal interpolation or smoothing of any kind.
- No vegetation index computation (done in Python later).
- No feature engineering.
- Index computation, cloud gap filling, EMA, and ALL feature engineering
  are handled externally in Python (pandas/numpy) for reproducibility and
  full control over causality.

CLOUD MASKING APPLIED IN GEE (§3.1)
-------------------------------------
SCL classes masked: 0 (no data), 1 (saturated), 2 (dark area / shadows),
3 (cloud shadow), 8 (cloud medium prob.), 9 (cloud high prob.),
10 (thin cirrus), 11 (snow/ice).
Morphological dilation: 1 × 10 m pixel expansion to catch cloud edges.
Masked pixels are set to NaN in the export.

USAGE
-----
1. Install the Earth Engine Python API:
       pip install earthengine-api

2. Authenticate (first run only):
       earthengine authenticate

3. Edit the CONFIGURATION block below (project ID, Drive folder, points CSV).

4. Run:
       python gee_extract.py

   Export tasks are submitted to GEE and run server-side.
   Monitor progress at https://code.earthengine.google.com/tasks

DEPENDENCIES
------------
    earthengine-api >= 0.1.370
    pandas >= 2.0
    (All other operations happen server-side on GEE.)
"""

import ee
import pandas as pd
import time
import math

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────────────────────────

GEE_PROJECT   = "abve-499717"       # GEE Cloud project ID
DRIVE_FOLDER  = "harvest_gee_exports"        # Destination folder in Google Drive
POINTS_CSV    = "../../data/sample_points.csv"          # Local CSV: columns [point_id, lat, lon]

# Temporal range (§2.1). Extend to 2019 if quota allows (§12, Optional Extension).
START_DATE    = "2022-05-01"
END_DATE      = "2025-12-15"
YEARS         = list(range(2022, 2026))      # [2022, 2023, 2024, 2025]

# S2 scene-level pre-filter: keep images with cloud cover below this threshold.
# Per-pixel SCL masking is the real filter; this just avoids downloading
# completely useless near-100%-cloudy images.
S2_MAX_CLOUD_PCT = 40

# Sentinel-1 pass direction (§2.2)
S1_PASS       = "DESCENDING"

# Scale for reduceRegions point extraction (metres).
# 10 m matches native resolution of both S1 and S2.
EXTRACT_SCALE = 10

# ─────────────────────────────────────────────────────────────────
# INITIALISE GEE
# ─────────────────────────────────────────────────────────────────

def init_gee(project: str) -> None:
    """Authenticate and initialise the Earth Engine Python API."""
    try:
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")
    except ee.EEException:
        print("  GEE credentials not found. Running ee.Authenticate() …")
        ee.Authenticate()
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")


# ─────────────────────────────────────────────────────────────────
# LOAD SAMPLE POINTS
# ─────────────────────────────────────────────────────────────────

def load_points(csv_path: str) -> ee.FeatureCollection:
    """
    Read a CSV of sample points and return a GEE FeatureCollection.

    Expected CSV columns: point_id (int), lat (float), lon (float).
    Any extra columns are preserved as Feature properties.
    """
    df = pd.read_csv(csv_path)
    required_cols = {"point_id", "lat", "lon"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"Points CSV must contain columns: {required_cols}. "
            f"Found: {list(df.columns)}"
        )

    features = []
    for _, row in df.iterrows():
        geom  = ee.Geometry.Point([float(row["lon"]), float(row["lat"])])
        props = {str(k): v for k, v in row.items()}
        features.append(ee.Feature(geom, props))

    fc = ee.FeatureCollection(features)
    print(f"✓ Loaded {len(df):,} sample points from '{csv_path}'")
    return fc


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — SCL CLOUD MASKING
# ─────────────────────────────────────────────────────────────────

def build_scl_mask(image: ee.Image) -> ee.Image:
    """
    Build a per-pixel cloud/shadow/snow mask from the SCL band (§3.1).

    SCL classes that are MASKED (set to NaN on export):
        0  — No data
        1  — Saturated / defective
        2  — Dark area pixels (cast shadows, dark soils)
        3  — Cloud shadow
        8  — Cloud medium probability
        9  — Cloud high probability
       10  — Thin cirrus
       11  — Snow / Ice

    SCL classes that are KEPT:
        4  — Vegetation
        5  — Not-vegetated
        6  — Water
        7  — Unclassified

    Morphological dilation (1 pixel / 10 m) is applied to the mask to
    remove contaminated cloud-edge pixels (§3.1, Criticism 11 adjudication).
    """
    scl = image.select("SCL")

    # Build a boolean mask: 1 = valid pixel, 0 = cloudy/shadow/snow
    invalid_classes = [0, 1, 2, 3, 8, 9, 10, 11]
    is_invalid = scl.eq(invalid_classes[0])
    for cls in invalid_classes[1:]:
        is_invalid = is_invalid.Or(scl.eq(cls))

    is_valid = is_invalid.Not()

    # Morphological erosion of the valid mask = dilation of the cloud mask.
    # focal_min with a 1-pixel (10 m) kernel shrinks the valid region by 1 pixel
    # around every cloud edge, effectively discarding contaminated border pixels.
    valid_dilated = is_valid.focal_min(radius=1, kernelType="square", units="pixels")

    return valid_dilated  # 1 = valid, 0 = masked


def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """Apply SCL-based cloud/shadow mask to a Sentinel-2 image."""
    mask = build_scl_mask(image)
    # updateMask sets masked pixels to NaN, which propagates to the CSV export
    return image.updateMask(mask)


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — BAND EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

# Raw bands to extract (§2.1).
# NOTE: SCL is NOT in this list — it is used for masking only and then dropped
# so the export contains only the radiometric surface-reflectance bands.
S2_BANDS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]


def extract_s2_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract raw S2 surface-reflectance bands
    for all sample points.

    Returns a FeatureCollection where each Feature = one point × one image date.
    Columns: point_id, lat, lon, date (YYYY-MM-DD), B2, B4, B5, B6, B7, B8,
             B11, B12.  Masked pixels are absent from the export (NaN in CSV).
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD_PCT))
        .select(S2_BANDS + ["SCL"])
        .map(mask_s2_clouds)
        .select(S2_BANDS)   # Drop SCL after masking
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        """reduceRegions over all points for a single image."""
        date_str = image.date().format("YYYY-MM-dd")

        reduced = image.reduceRegions(
            collection  = points,
            reducer     = ee.Reducer.mean(),  # Mean over the ~10 m buffer
            scale       = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S2_BANDS))

        # Attach the image acquisition date to every point feature
        return reduced.map(lambda f: f.set("date", date_str))
    
    # Map over the entire collection → flat FeatureCollection of (point × date) rows
    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# SENTINEL-1 — SAR BACKSCATTER EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

S1_BANDS = ["VH", "VV"]


def extract_s1_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract Sentinel-1 GRD VH and VV backscatter
    (in dB) for all sample points.

    Filters applied (§2.2):
      - Instrument mode: IW (Interferometric Wide)
      - Pass direction: DESCENDING (consistent geometry across dates)
      - Bands: VH, VV

    Returns FeatureCollection: point_id, date, VH, VV.
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", S1_PASS))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select(S1_BANDS)
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        date_str = image.date().format("YYYY-MM-dd")
        reduced  = image.reduceRegions(
            collection = points,
            reducer    = ee.Reducer.mean(),
            scale      = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S1_BANDS))
        return reduced.map(lambda f: f.set("date", date_str))

    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# STATIC LAYERS — SRTM + ESA WORLDCOVER (one-time extract)
# ─────────────────────────────────────────────────────────────────

def extract_static_layers(points: ee.FeatureCollection) -> ee.FeatureCollection:
    """
    Extract static (time-invariant) geographic layers for each sample point.

    Layers extracted:
      - SRTM v4.1 @ 30 m: elevation (m), slope (°), aspect (°)
      - ESA WorldCover 2021 @ 10 m: land cover class integer
        (10 = tree cover, 40 = cropland, etc.)

    These are fetched once and merged with the time-series data during the
    Python feature engineering step.
    """
    # SRTM elevation + derived terrain metrics
    srtm      = ee.Image("USGS/SRTMGL1_003")
    elevation = srtm.select("elevation")
    slope     = ee.Terrain.slope(elevation)
    aspect    = ee.Terrain.aspect(elevation)

    terrain = elevation.rename("elevation_m") \
                       .addBands(slope.rename("slope_deg")) \
                       .addBands(aspect.rename("aspect_deg"))

    # ESA WorldCover 2021 (10 m)
    worldcover = (
        ee.ImageCollection("ESA/WorldCover/v200")
        .first()
        .select("Map")
        .rename("worldcover_class")
    )

    static_image = terrain.addBands(worldcover)

    static_fc = static_image.reduceRegions(
        collection = points,
        reducer    = ee.Reducer.mean(),
        scale      = 30,   # SRTM native resolution
    )

    return static_fc


# ─────────────────────────────────────────────────────────────────
# COLUMN CLEANUP — SELECT ONLY REQUIRED COLUMNS FOR EXPORT
# ─────────────────────────────────────────────────────────────────

def select_s2_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S2 CSV."""
    keep = ["point_id", "lat", "lon"] + S2_BANDS + ["date"]
    return fc.select(keep)


def select_s1_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S1 CSV."""
    keep = ["point_id", "lat", "lon"] + S1_BANDS + ["date"]
    return fc.select(keep)


def select_static_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the static CSV."""
    keep = ["point_id", "lat", "lon",
            "elevation_m", "slope_deg", "aspect_deg", "worldcover_class"]
    return fc.select(keep)


# ─────────────────────────────────────────────────────────────────
# EXPORT HELPERS
# ─────────────────────────────────────────────────────────────────

def export_to_drive(
    fc: ee.FeatureCollection,
    description: str,
    folder: str,
    filename: str,
) -> ee.batch.Task:
    """
    Submit a GEE Export.table.toDrive task for a FeatureCollection.

    Returns the Task object (already started). Monitor at
    https://code.earthengine.google.com/tasks
    """
    task = ee.batch.Export.table.toDrive(
        collection    = fc,
        description   = description,
        folder        = folder,
        fileNamePrefix= filename,
        fileFormat    = "CSV",
    )
    task.start()
    print(f"  → Task submitted: '{description}'  (filename: {filename}.csv)")
    return task


def wait_for_tasks(tasks: list, poll_interval_s: int = 30) -> None:
    """
    Poll all submitted tasks until they complete or fail.
    Optional — you can also just let them run and monitor on the GEE Tasks page.
    """
    print("\n⏳  Polling task status (Ctrl+C to stop polling without cancelling tasks) …")
    remaining = {t.id: t for t in tasks}

    while remaining:
        time.sleep(poll_interval_s)
        done = []
        for tid, task in remaining.items():
            status = task.status()
            state  = status["state"]
            name   = status.get("description", tid)
            if state in ("COMPLETED", "FAILED", "CANCELLED"):
                icon = "✓" if state == "COMPLETED" else "✗"
                print(f"  {icon} [{state}] {name}")
                done.append(tid)
        for tid in done:
            del remaining[tid]

    print("✓ All tasks finished.")


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def main() -> None:
    # ── 1. Initialise ────────────────────────────────────────────
    init_gee(GEE_PROJECT)

    # ── 2. Load points ───────────────────────────────────────────
    points = load_points(POINTS_CSV)

    submitted_tasks = []

    # ── 3. Sentinel-2 export — one task per year ─────────────────
    print("\n── Sentinel-2 raw band extraction ──────────────────────────")
    for year in YEARS:
        print(f"  Processing S2 year {year} …")
        fc       = extract_s2_year(points, year)
        fc_clean = select_s2_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S2_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel2_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 4. Sentinel-1 export — one task per year ─────────────────
    print("\n── Sentinel-1 SAR backscatter extraction ───────────────────")
    for year in YEARS:
        print(f"  Processing S1 year {year} …")
        fc       = extract_s1_year(points, year)
        fc_clean = select_s1_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S1_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel1_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 5. Static layers export — one-time ───────────────────────
    print("\n── Static layers (SRTM + WorldCover) ───────────────────────")
    fc_static  = extract_static_layers(points)
    fc_static_clean = select_static_columns(fc_static)
    task_static = export_to_drive(
        fc          = fc_static_clean,
        description = "static_layers",
        folder      = DRIVE_FOLDER,
        filename    = "static_layers",
    )
    submitted_tasks.append(task_static)

    # ── 6. Summary ───────────────────────────────────────────────
    total = len(submitted_tasks)
    print(f"\n✓ {total} export tasks submitted to GEE.")
    print(f"  Files will appear in Google Drive → '{DRIVE_FOLDER}/' once complete.")
    print("  Monitor progress at: https://code.earthengine.google.com/tasks\n")

    print("Expected output files:")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv")
    print(f"  {DRIVE_FOLDER}/static_layers.csv")

    # ── 7. Optional: block and poll until all tasks finish ────────
    # Uncomment the line below if you want the script to wait and
    # print live status updates.  Otherwise tasks run in background.
    # wait_for_tasks(submitted_tasks, poll_interval_s=30)


if __name__ == "__main__":
    main()

c:\Users\nishk\anaconda3\envs\abve\Lib\site-packages\ee\deprecation.py:140: UserWarning: Unable to initialize deprecated assets: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4057)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


✓ GEE initialised — project: abve-499717
✓ Loaded 829 sample points from '../../data/sample_points.csv'

── Sentinel-2 raw band extraction ──────────────────────────
  Processing S2 year 2022 …
  → Task submitted: 'S2_raw_2022'  (filename: sentinel2_raw_2022.csv)
  Processing S2 year 2023 …
  → Task submitted: 'S2_raw_2023'  (filename: sentinel2_raw_2023.csv)
  Processing S2 year 2024 …
  → Task submitted: 'S2_raw_2024'  (filename: sentinel2_raw_2024.csv)
  Processing S2 year 2025 …
  → Task submitted: 'S2_raw_2025'  (filename: sentinel2_raw_2025.csv)

── Sentinel-1 SAR backscatter extraction ───────────────────
  Processing S1 year 2022 …
  → Task submitted: 'S1_raw_2022'  (filename: sentinel1_raw_2022.csv)
  Processing S1 year 2023 …
  → Task submitted: 'S1_raw_2023'  (filename: sentinel1_raw_2023.csv)
  Processing S1 year 2024 …
  → Task submitted: 'S1_raw_2024'  (filename: sentinel1_raw_2024.csv)
  Processing S1 year 2025 …
  → Task submitted: 'S1_raw_2025'  (filename: sentinel1_ra

In [ ]:
S1A_IW_GRDH_1SDV_20220501T004413_20220501T004438_043010_0522A0_719A_0

for [2018, 2021]

In [1]:
"""
gee_extract.py
==============
Google Earth Engine — Raw Data Extraction Script
Harvest Date Prediction Pipeline (Implementation Plan v2)

WHAT THIS SCRIPT DOES
----------------------
1. Authenticates with GEE and initialises the project.
2. Ingests a user-supplied CSV of 1,000 sample points (point_id, lat, lon).
3. Exports TWO separate CSVs to Google Drive (one per satellite source):

   a) sentinel2_raw_<year>.csv  — raw S2 bands (B2,B4,B5,B6,B7,B8,B11,B12) +
                                   SCL cloud mask, per point × date.
   b) sentinel1_raw_<year>.csv  — S1 GRD VH and VV backscatter (dB),
                                   per point × date.

NOTE: static_layers.csv is intentionally NOT re-exported in this run.
      It already exists at harvest_gee_exports/static_layers.csv (produced
      during the 2022-2025 extraction run) and is time-invariant — re-exporting
      would overwrite it with an identical file.  The existing file is safe to
      reuse as-is for the 2018-2021 data.

WHAT THIS SCRIPT DOES NOT DO (by design — see §12, Milestone 0)
-----------------------------------------------------------------
- No temporal interpolation or smoothing of any kind.
- No vegetation index computation (done in Python later).
- No feature engineering.
- Index computation, cloud gap filling, EMA, and ALL feature engineering
  are handled externally in Python (pandas/numpy) for reproducibility and
  full control over causality.

CLOUD MASKING APPLIED IN GEE (§3.1)
-------------------------------------
SCL classes masked: 0 (no data), 1 (saturated), 2 (dark area / shadows),
3 (cloud shadow), 8 (cloud medium prob.), 9 (cloud high prob.),
10 (thin cirrus), 11 (snow/ice).
Morphological dilation: 1 × 10 m pixel expansion to catch cloud edges.
Masked pixels are set to NaN in the export.

SAFE COEXISTENCE WITH EXISTING 2022-2025 EXPORTS
-------------------------------------------------
This run exports ONLY years [2018, 2019, 2020, 2021].  The output filenames
are year-stamped (e.g. sentinel2_raw_2018.csv), so they will never collide
with or overwrite the existing files below:
    harvest_gee_exports/sentinel2_raw_2022.csv
    harvest_gee_exports/sentinel2_raw_2023.csv
    harvest_gee_exports/sentinel2_raw_2024.csv
    harvest_gee_exports/sentinel2_raw_2025.csv
    harvest_gee_exports/sentinel1_raw_2022.csv
    harvest_gee_exports/sentinel1_raw_2023.csv
    harvest_gee_exports/sentinel1_raw_2024.csv
    harvest_gee_exports/sentinel1_raw_2025.csv
    harvest_gee_exports/static_layers.csv   ← skipped entirely in this run

USAGE
-----
1. Install the Earth Engine Python API:
       pip install earthengine-api

2. Authenticate (first run only):
       earthengine authenticate

3. Edit the CONFIGURATION block below (project ID, Drive folder, points CSV).

4. Run:
       python gee_extract.py

   Export tasks are submitted to GEE and run server-side.
   Monitor progress at https://code.earthengine.google.com/tasks

DEPENDENCIES
------------
    earthengine-api >= 0.1.370
    pandas >= 2.0
    (All other operations happen server-side on GEE.)
"""

import ee
import pandas as pd
import time
import math

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────────────────────────

GEE_PROJECT   = "abve-499717"                  # GEE Cloud project ID
DRIVE_FOLDER  = "harvest_gee_exports"          # Destination folder in Google Drive
POINTS_CSV    = "../../data/sample_points.csv" # Local CSV: columns [point_id, lat, lon]

# ── Temporal range for THIS run ───────────────────────────────────
# Changed from [2022-2025] to [2018-2021].
# Output filenames are year-stamped, so existing 2022-2025 CSVs in
# harvest_gee_exports/ are completely unaffected.
START_DATE    = "2018-05-01"
END_DATE      = "2021-12-15"
YEARS         = [2018, 2019, 2020, 2021]       # ← updated

# S2 scene-level pre-filter: keep images with cloud cover below this threshold.
# Per-pixel SCL masking is the real filter; this just avoids downloading
# completely useless near-100%-cloudy images.
S2_MAX_CLOUD_PCT = 40

# Sentinel-1 pass direction (§2.2)
S1_PASS       = "DESCENDING"

# Scale for reduceRegions point extraction (metres).
# 10 m matches native resolution of both S1 and S2.
EXTRACT_SCALE = 10

# ─────────────────────────────────────────────────────────────────
# INITIALISE GEE
# ─────────────────────────────────────────────────────────────────

def init_gee(project: str) -> None:
    """Authenticate and initialise the Earth Engine Python API."""
    try:
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")
    except ee.EEException:
        print("  GEE credentials not found. Running ee.Authenticate() …")
        ee.Authenticate()
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")


# ─────────────────────────────────────────────────────────────────
# LOAD SAMPLE POINTS
# ─────────────────────────────────────────────────────────────────

def load_points(csv_path: str) -> ee.FeatureCollection:
    """
    Read a CSV of sample points and return a GEE FeatureCollection.

    Expected CSV columns: point_id (int), lat (float), lon (float).
    Any extra columns are preserved as Feature properties.
    """
    df = pd.read_csv(csv_path)
    required_cols = {"point_id", "lat", "lon"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"Points CSV must contain columns: {required_cols}. "
            f"Found: {list(df.columns)}"
        )

    features = []
    for _, row in df.iterrows():
        geom  = ee.Geometry.Point([float(row["lon"]), float(row["lat"])])
        props = {str(k): v for k, v in row.items()}
        features.append(ee.Feature(geom, props))

    fc = ee.FeatureCollection(features)
    print(f"✓ Loaded {len(df):,} sample points from '{csv_path}'")
    return fc


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — SCL CLOUD MASKING
# ─────────────────────────────────────────────────────────────────

def build_scl_mask(image: ee.Image) -> ee.Image:
    """
    Build a per-pixel cloud/shadow/snow mask from the SCL band (§3.1).

    SCL classes that are MASKED (set to NaN on export):
        0  — No data
        1  — Saturated / defective
        2  — Dark area pixels (cast shadows, dark soils)
        3  — Cloud shadow
        8  — Cloud medium probability
        9  — Cloud high probability
       10  — Thin cirrus
       11  — Snow / Ice

    SCL classes that are KEPT:
        4  — Vegetation
        5  — Not-vegetated
        6  — Water
        7  — Unclassified

    Morphological dilation (1 pixel / 10 m) is applied to the mask to
    remove contaminated cloud-edge pixels (§3.1, Criticism 11 adjudication).
    """
    scl = image.select("SCL")

    # Build a boolean mask: 1 = valid pixel, 0 = cloudy/shadow/snow
    invalid_classes = [0, 1, 2, 3, 8, 9, 10, 11]
    is_invalid = scl.eq(invalid_classes[0])
    for cls in invalid_classes[1:]:
        is_invalid = is_invalid.Or(scl.eq(cls))

    is_valid = is_invalid.Not()

    # Morphological erosion of the valid mask = dilation of the cloud mask.
    # focal_min with a 1-pixel (10 m) kernel shrinks the valid region by 1 pixel
    # around every cloud edge, effectively discarding contaminated border pixels.
    valid_dilated = is_valid.focal_min(radius=1, kernelType="square", units="pixels")

    return valid_dilated  # 1 = valid, 0 = masked


def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """Apply SCL-based cloud/shadow mask to a Sentinel-2 image."""
    mask = build_scl_mask(image)
    # updateMask sets masked pixels to NaN, which propagates to the CSV export
    return image.updateMask(mask)


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — BAND EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

# Raw bands to extract (§2.1).
# NOTE: SCL is NOT in this list — it is used for masking only and then dropped
# so the export contains only the radiometric surface-reflectance bands.
S2_BANDS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]


def extract_s2_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract raw S2 surface-reflectance bands
    for all sample points.

    Returns a FeatureCollection where each Feature = one point × one image date.
    Columns: point_id, lat, lon, date (YYYY-MM-DD), B2, B4, B5, B6, B7, B8,
             B11, B12.  Masked pixels are absent from the export (NaN in CSV).
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD_PCT))
        .select(S2_BANDS + ["SCL"])
        .map(mask_s2_clouds)
        .select(S2_BANDS)   # Drop SCL after masking
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        """reduceRegions over all points for a single image."""
        date_str = image.date().format("YYYY-MM-dd")

        reduced = image.reduceRegions(
            collection  = points,
            reducer     = ee.Reducer.mean(),  # Mean over the ~10 m buffer
            scale       = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S2_BANDS))

        # Attach the image acquisition date to every point feature
        return reduced.map(lambda f: f.set("date", date_str))

    # Map over the entire collection → flat FeatureCollection of (point × date) rows
    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# SENTINEL-1 — SAR BACKSCATTER EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

S1_BANDS = ["VH", "VV"]


def extract_s1_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract Sentinel-1 GRD VH and VV backscatter
    (in dB) for all sample points.

    Filters applied (§2.2):
      - Instrument mode: IW (Interferometric Wide)
      - Pass direction: DESCENDING (consistent geometry across dates)
      - Bands: VH, VV

    Returns FeatureCollection: point_id, date, VH, VV.
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", S1_PASS))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select(S1_BANDS)
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        date_str = image.date().format("YYYY-MM-dd")
        reduced  = image.reduceRegions(
            collection = points,
            reducer    = ee.Reducer.mean(),
            scale      = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S1_BANDS))
        return reduced.map(lambda f: f.set("date", date_str))

    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# COLUMN CLEANUP — SELECT ONLY REQUIRED COLUMNS FOR EXPORT
# ─────────────────────────────────────────────────────────────────

def select_s2_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S2 CSV."""
    keep = ["point_id", "lat", "lon"] + S2_BANDS + ["date"]
    return fc.select(keep)


def select_s1_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S1 CSV."""
    keep = ["point_id", "lat", "lon"] + S1_BANDS + ["date"]
    return fc.select(keep)


# ─────────────────────────────────────────────────────────────────
# EXPORT HELPERS
# ─────────────────────────────────────────────────────────────────

def export_to_drive(
    fc: ee.FeatureCollection,
    description: str,
    folder: str,
    filename: str,
) -> ee.batch.Task:
    """
    Submit a GEE Export.table.toDrive task for a FeatureCollection.

    Returns the Task object (already started). Monitor at
    https://code.earthengine.google.com/tasks
    """
    task = ee.batch.Export.table.toDrive(
        collection    = fc,
        description   = description,
        folder        = folder,
        fileNamePrefix= filename,
        fileFormat    = "CSV",
    )
    task.start()
    print(f"  → Task submitted: '{description}'  (filename: {filename}.csv)")
    return task


def wait_for_tasks(tasks: list, poll_interval_s: int = 30) -> None:
    """
    Poll all submitted tasks until they complete or fail.
    Optional — you can also just let them run and monitor on the GEE Tasks page.
    """
    print("\n⏳  Polling task status (Ctrl+C to stop polling without cancelling tasks) …")
    remaining = {t.id: t for t in tasks}

    while remaining:
        time.sleep(poll_interval_s)
        done = []
        for tid, task in remaining.items():
            status = task.status()
            state  = status["state"]
            name   = status.get("description", tid)
            if state in ("COMPLETED", "FAILED", "CANCELLED"):
                icon = "✓" if state == "COMPLETED" else "✗"
                print(f"  {icon} [{state}] {name}")
                done.append(tid)
        for tid in done:
            del remaining[tid]

    print("✓ All tasks finished.")


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def main() -> None:
    # ── 1. Initialise ────────────────────────────────────────────
    init_gee(GEE_PROJECT)

    # ── 2. Load points ───────────────────────────────────────────
    points = load_points(POINTS_CSV)

    submitted_tasks = []

    # ── 3. Sentinel-2 export — one task per year ─────────────────
    print("\n── Sentinel-2 raw band extraction ──────────────────────────")
    for year in YEARS:
        print(f"  Processing S2 year {year} …")
        fc       = extract_s2_year(points, year)
        fc_clean = select_s2_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S2_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel2_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 4. Sentinel-1 export — one task per year ─────────────────
    print("\n── Sentinel-1 SAR backscatter extraction ───────────────────")
    for year in YEARS:
        print(f"  Processing S1 year {year} …")
        fc       = extract_s1_year(points, year)
        fc_clean = select_s1_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S1_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel1_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 5. Static layers — SKIPPED ───────────────────────────────
    # static_layers.csv already exists from the 2022-2025 run and is
    # time-invariant (SRTM + WorldCover do not change year to year).
    # Re-exporting would risk overwriting the existing file with no benefit.
    # Use harvest_gee_exports/static_layers.csv as-is for all years.
    print("\n── Static layers (SRTM + WorldCover) ───────────────────────")
    print("  SKIPPED — static_layers.csv already exists in Drive and is")
    print("  time-invariant. Reuse the existing file for 2018-2021 data.")

    # ── 6. Summary ───────────────────────────────────────────────
    total = len(submitted_tasks)
    print(f"\n✓ {total} export tasks submitted to GEE.")
    print(f"  Files will appear in Google Drive → '{DRIVE_FOLDER}/' once complete.")
    print("  Monitor progress at: https://code.earthengine.google.com/tasks\n")

    print("Expected NEW output files (existing 2022-2025 files are untouched):")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv")
    print(f"\nExisting files preserved:")
    for year in [2022, 2023, 2024, 2025]:
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv  ← untouched")
    for year in [2022, 2023, 2024, 2025]:
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv  ← untouched")
    print(f"  {DRIVE_FOLDER}/static_layers.csv              ← untouched (skipped)")

    # ── 7. Optional: block and poll until all tasks finish ────────
    # Uncomment the line below if you want the script to wait and
    # print live status updates.  Otherwise tasks run in background.
    # wait_for_tasks(submitted_tasks, poll_interval_s=30)


if __name__ == "__main__":
    main()

c:\Users\nishk\anaconda3\envs\abve\Lib\site-packages\ee\deprecation.py:140: UserWarning: Unable to initialize deprecated assets: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4057)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


✓ GEE initialised — project: abve-499717
✓ Loaded 829 sample points from '../../data/sample_points.csv'

── Sentinel-2 raw band extraction ──────────────────────────
  Processing S2 year 2018 …
  → Task submitted: 'S2_raw_2018'  (filename: sentinel2_raw_2018.csv)
  Processing S2 year 2019 …
  → Task submitted: 'S2_raw_2019'  (filename: sentinel2_raw_2019.csv)
  Processing S2 year 2020 …
  → Task submitted: 'S2_raw_2020'  (filename: sentinel2_raw_2020.csv)
  Processing S2 year 2021 …
  → Task submitted: 'S2_raw_2021'  (filename: sentinel2_raw_2021.csv)

── Sentinel-1 SAR backscatter extraction ───────────────────
  Processing S1 year 2018 …
  → Task submitted: 'S1_raw_2018'  (filename: sentinel1_raw_2018.csv)
  Processing S1 year 2019 …
  → Task submitted: 'S1_raw_2019'  (filename: sentinel1_raw_2019.csv)
  Processing S1 year 2020 …
  → Task submitted: 'S1_raw_2020'  (filename: sentinel1_raw_2020.csv)
  Processing S1 year 2021 …
  → Task submitted: 'S1_raw_2021'  (filename: sentinel1_ra

2017 too 👉👈

In [2]:
"""
gee_extract.py
==============
Google Earth Engine — Raw Data Extraction Script
Harvest Date Prediction Pipeline (Implementation Plan v2)

WHAT THIS SCRIPT DOES
----------------------
1. Authenticates with GEE and initialises the project.
2. Ingests a user-supplied CSV of 1,000 sample points (point_id, lat, lon).
3. Exports TWO separate CSVs to Google Drive (one per satellite source):

   a) sentinel2_raw_<year>.csv  — raw S2 bands (B2,B4,B5,B6,B7,B8,B11,B12) +
                                   SCL cloud mask, per point × date.
   b) sentinel1_raw_<year>.csv  — S1 GRD VH and VV backscatter (dB),
                                   per point × date.

NOTE: static_layers.csv is intentionally NOT re-exported in this run.
      It already exists at harvest_gee_exports/static_layers.csv (produced
      during the 2022-2025 extraction run) and is time-invariant — re-exporting
      would overwrite it with an identical file.  The existing file is safe to
      reuse as-is for 2017 data too.

WHAT THIS SCRIPT DOES NOT DO (by design — see §12, Milestone 0)
-----------------------------------------------------------------
- No temporal interpolation or smoothing of any kind.
- No vegetation index computation (done in Python later).
- No feature engineering.
- Index computation, cloud gap filling, EMA, and ALL feature engineering
  are handled externally in Python (pandas/numpy) for reproducibility and
  full control over causality.

CLOUD MASKING APPLIED IN GEE (§3.1)
-------------------------------------
SCL classes masked: 0 (no data), 1 (saturated), 2 (dark area / shadows),
3 (cloud shadow), 8 (cloud medium prob.), 9 (cloud high prob.),
10 (thin cirrus), 11 (snow/ice).
Morphological dilation: 1 × 10 m pixel expansion to catch cloud edges.
Masked pixels are set to NaN in the export.

SAFE COEXISTENCE WITH EXISTING 2018-2025 EXPORTS
-------------------------------------------------
This run exports ONLY year [2017].  The output filenames are year-stamped
(sentinel2_raw_2017.csv, sentinel1_raw_2017.csv), so they will never collide
with or overwrite any of the existing files below:
    harvest_gee_exports/sentinel2_raw_2018.csv through sentinel2_raw_2025.csv
    harvest_gee_exports/sentinel1_raw_2018.csv through sentinel1_raw_2025.csv
    harvest_gee_exports/static_layers.csv   ← skipped entirely in this run

USAGE
-----
1. Install the Earth Engine Python API:
       pip install earthengine-api

2. Authenticate (first run only):
       earthengine authenticate

3. Edit the CONFIGURATION block below (project ID, Drive folder, points CSV).

4. Run:
       python gee_extract.py

   Export tasks are submitted to GEE and run server-side.
   Monitor progress at https://code.earthengine.google.com/tasks

DEPENDENCIES
------------
    earthengine-api >= 0.1.370
    pandas >= 2.0
    (All other operations happen server-side on GEE.)
"""

import ee
import pandas as pd
import time
import math

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────────────────────────

GEE_PROJECT   = "abve-499717"                  # GEE Cloud project ID
DRIVE_FOLDER  = "harvest_gee_exports"          # Destination folder in Google Drive
POINTS_CSV    = "../../data/sample_points.csv" # Local CSV: columns [point_id, lat, lon]

# ── Temporal range for THIS run ───────────────────────────────────
# Targeting 2017 only.  All existing 2018-2025 CSVs in
# harvest_gee_exports/ are completely unaffected (different filenames).
START_DATE    = "2017-05-01"
END_DATE      = "2017-12-15"
YEARS         = [2017]                         # ← 2017 only

# S2 scene-level pre-filter: keep images with cloud cover below this threshold.
# Per-pixel SCL masking is the real filter; this just avoids downloading
# completely useless near-100%-cloudy images.
S2_MAX_CLOUD_PCT = 40

# Sentinel-1 pass direction (§2.2)
S1_PASS       = "DESCENDING"

# Scale for reduceRegions point extraction (metres).
# 10 m matches native resolution of both S1 and S2.
EXTRACT_SCALE = 10

# ─────────────────────────────────────────────────────────────────
# INITIALISE GEE
# ─────────────────────────────────────────────────────────────────

def init_gee(project: str) -> None:
    """Authenticate and initialise the Earth Engine Python API."""
    try:
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")
    except ee.EEException:
        print("  GEE credentials not found. Running ee.Authenticate() …")
        ee.Authenticate()
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")


# ─────────────────────────────────────────────────────────────────
# LOAD SAMPLE POINTS
# ─────────────────────────────────────────────────────────────────

def load_points(csv_path: str) -> ee.FeatureCollection:
    """
    Read a CSV of sample points and return a GEE FeatureCollection.

    Expected CSV columns: point_id (int), lat (float), lon (float).
    Any extra columns are preserved as Feature properties.
    """
    df = pd.read_csv(csv_path)
    required_cols = {"point_id", "lat", "lon"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"Points CSV must contain columns: {required_cols}. "
            f"Found: {list(df.columns)}"
        )

    features = []
    for _, row in df.iterrows():
        geom  = ee.Geometry.Point([float(row["lon"]), float(row["lat"])])
        props = {str(k): v for k, v in row.items()}
        features.append(ee.Feature(geom, props))

    fc = ee.FeatureCollection(features)
    print(f"✓ Loaded {len(df):,} sample points from '{csv_path}'")
    return fc


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — SCL CLOUD MASKING
# ─────────────────────────────────────────────────────────────────

def build_scl_mask(image: ee.Image) -> ee.Image:
    """
    Build a per-pixel cloud/shadow/snow mask from the SCL band (§3.1).

    SCL classes that are MASKED (set to NaN on export):
        0  — No data
        1  — Saturated / defective
        2  — Dark area pixels (cast shadows, dark soils)
        3  — Cloud shadow
        8  — Cloud medium probability
        9  — Cloud high probability
       10  — Thin cirrus
       11  — Snow / Ice

    SCL classes that are KEPT:
        4  — Vegetation
        5  — Not-vegetated
        6  — Water
        7  — Unclassified

    Morphological dilation (1 pixel / 10 m) is applied to the mask to
    remove contaminated cloud-edge pixels (§3.1, Criticism 11 adjudication).
    """
    scl = image.select("SCL")

    # Build a boolean mask: 1 = valid pixel, 0 = cloudy/shadow/snow
    invalid_classes = [0, 1, 2, 3, 8, 9, 10, 11]
    is_invalid = scl.eq(invalid_classes[0])
    for cls in invalid_classes[1:]:
        is_invalid = is_invalid.Or(scl.eq(cls))

    is_valid = is_invalid.Not()

    # Morphological erosion of the valid mask = dilation of the cloud mask.
    # focal_min with a 1-pixel (10 m) kernel shrinks the valid region by 1 pixel
    # around every cloud edge, effectively discarding contaminated border pixels.
    valid_dilated = is_valid.focal_min(radius=1, kernelType="square", units="pixels")

    return valid_dilated  # 1 = valid, 0 = masked


def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """Apply SCL-based cloud/shadow mask to a Sentinel-2 image."""
    mask = build_scl_mask(image)
    # updateMask sets masked pixels to NaN, which propagates to the CSV export
    return image.updateMask(mask)


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — BAND EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

# Raw bands to extract (§2.1).
# NOTE: SCL is NOT in this list — it is used for masking only and then dropped
# so the export contains only the radiometric surface-reflectance bands.
S2_BANDS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]


def extract_s2_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract raw S2 surface-reflectance bands
    for all sample points.

    Returns a FeatureCollection where each Feature = one point × one image date.
    Columns: point_id, lat, lon, date (YYYY-MM-DD), B2, B4, B5, B6, B7, B8,
             B11, B12.  Masked pixels are absent from the export (NaN in CSV).
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD_PCT))
        .select(S2_BANDS + ["SCL"])
        .map(mask_s2_clouds)
        .select(S2_BANDS)   # Drop SCL after masking
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        """reduceRegions over all points for a single image."""
        date_str = image.date().format("YYYY-MM-dd")

        reduced = image.reduceRegions(
            collection  = points,
            reducer     = ee.Reducer.mean(),  # Mean over the ~10 m buffer
            scale       = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S2_BANDS))

        # Attach the image acquisition date to every point feature
        return reduced.map(lambda f: f.set("date", date_str))

    # Map over the entire collection → flat FeatureCollection of (point × date) rows
    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# SENTINEL-1 — SAR BACKSCATTER EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

S1_BANDS = ["VH", "VV"]


def extract_s1_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract Sentinel-1 GRD VH and VV backscatter
    (in dB) for all sample points.

    Filters applied (§2.2):
      - Instrument mode: IW (Interferometric Wide)
      - Pass direction: DESCENDING (consistent geometry across dates)
      - Bands: VH, VV

    Returns FeatureCollection: point_id, date, VH, VV.
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", S1_PASS))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select(S1_BANDS)
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        date_str = image.date().format("YYYY-MM-dd")
        reduced  = image.reduceRegions(
            collection = points,
            reducer    = ee.Reducer.mean(),
            scale      = EXTRACT_SCALE
        ).filter(ee.Filter.notNull(S1_BANDS))
        return reduced.map(lambda f: f.set("date", date_str))

    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# COLUMN CLEANUP — SELECT ONLY REQUIRED COLUMNS FOR EXPORT
# ─────────────────────────────────────────────────────────────────

def select_s2_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S2 CSV."""
    keep = ["point_id", "lat", "lon"] + S2_BANDS + ["date"]
    return fc.select(keep)


def select_s1_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S1 CSV."""
    keep = ["point_id", "lat", "lon"] + S1_BANDS + ["date"]
    return fc.select(keep)


# ─────────────────────────────────────────────────────────────────
# EXPORT HELPERS
# ─────────────────────────────────────────────────────────────────

def export_to_drive(
    fc: ee.FeatureCollection,
    description: str,
    folder: str,
    filename: str,
) -> ee.batch.Task:
    """
    Submit a GEE Export.table.toDrive task for a FeatureCollection.

    Returns the Task object (already started). Monitor at
    https://code.earthengine.google.com/tasks
    """
    task = ee.batch.Export.table.toDrive(
        collection    = fc,
        description   = description,
        folder        = folder,
        fileNamePrefix= filename,
        fileFormat    = "CSV",
    )
    task.start()
    print(f"  → Task submitted: '{description}'  (filename: {filename}.csv)")
    return task


def wait_for_tasks(tasks: list, poll_interval_s: int = 30) -> None:
    """
    Poll all submitted tasks until they complete or fail.
    Optional — you can also just let them run and monitor on the GEE Tasks page.
    """
    print("\n⏳  Polling task status (Ctrl+C to stop polling without cancelling tasks) …")
    remaining = {t.id: t for t in tasks}

    while remaining:
        time.sleep(poll_interval_s)
        done = []
        for tid, task in remaining.items():
            status = task.status()
            state  = status["state"]
            name   = status.get("description", tid)
            if state in ("COMPLETED", "FAILED", "CANCELLED"):
                icon = "✓" if state == "COMPLETED" else "✗"
                print(f"  {icon} [{state}] {name}")
                done.append(tid)
        for tid in done:
            del remaining[tid]

    print("✓ All tasks finished.")


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def main() -> None:
    # ── 1. Initialise ────────────────────────────────────────────
    init_gee(GEE_PROJECT)

    # ── 2. Load points ───────────────────────────────────────────
    points = load_points(POINTS_CSV)

    submitted_tasks = []

    # ── 3. Sentinel-2 export — one task per year ─────────────────
    print("\n── Sentinel-2 raw band extraction ──────────────────────────")
    for year in YEARS:
        print(f"  Processing S2 year {year} …")
        fc       = extract_s2_year(points, year)
        fc_clean = select_s2_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S2_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel2_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 4. Sentinel-1 export — one task per year ─────────────────
    print("\n── Sentinel-1 SAR backscatter extraction ───────────────────")
    for year in YEARS:
        print(f"  Processing S1 year {year} …")
        fc       = extract_s1_year(points, year)
        fc_clean = select_s1_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S1_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel1_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 5. Static layers — SKIPPED ───────────────────────────────
    # static_layers.csv already exists from the 2022-2025 run and is
    # time-invariant (SRTM + WorldCover do not change year to year).
    # Re-exporting would risk overwriting the existing file with no benefit.
    # Use harvest_gee_exports/static_layers.csv as-is for all years.
    print("\n── Static layers (SRTM + WorldCover) ───────────────────────")
    print("  SKIPPED — static_layers.csv already exists in Drive and is")
    print("  time-invariant. Reuse the existing file for 2017 data too.")

    # ── 6. Summary ───────────────────────────────────────────────
    total = len(submitted_tasks)
    print(f"\n✓ {total} export tasks submitted to GEE.")
    print(f"  Files will appear in Google Drive → '{DRIVE_FOLDER}/' once complete.")
    print("  Monitor progress at: https://code.earthengine.google.com/tasks\n")

    print("Expected NEW output files (all existing 2018-2025 files are untouched):")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv")
    print(f"\nExisting files preserved:")
    for year in range(2018, 2026):
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv  ← untouched")
    for year in range(2018, 2026):
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv  ← untouched")
    print(f"  {DRIVE_FOLDER}/static_layers.csv              ← untouched (skipped)")

    # ── 7. Optional: block and poll until all tasks finish ────────
    # Uncomment the line below if you want the script to wait and
    # print live status updates.  Otherwise tasks run in background.
    # wait_for_tasks(submitted_tasks, poll_interval_s=30)


if __name__ == "__main__":
    main()

c:\Users\nishk\anaconda3\envs\abve\Lib\site-packages\ee\deprecation.py:140: UserWarning: Unable to initialize deprecated assets: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4057)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


✓ GEE initialised — project: abve-499717
✓ Loaded 829 sample points from '../../data/sample_points.csv'

── Sentinel-2 raw band extraction ──────────────────────────
  Processing S2 year 2017 …
  → Task submitted: 'S2_raw_2017'  (filename: sentinel2_raw_2017.csv)

── Sentinel-1 SAR backscatter extraction ───────────────────
  Processing S1 year 2017 …
  → Task submitted: 'S1_raw_2017'  (filename: sentinel1_raw_2017.csv)

── Static layers (SRTM + WorldCover) ───────────────────────
  SKIPPED — static_layers.csv already exists in Drive and is
  time-invariant. Reuse the existing file for 2017 data too.

✓ 2 export tasks submitted to GEE.
  Files will appear in Google Drive → 'harvest_gee_exports/' once complete.
  Monitor progress at: https://code.earthengine.google.com/tasks

Expected NEW output files (all existing 2018-2025 files are untouched):
  harvest_gee_exports/sentinel2_raw_2017.csv
  harvest_gee_exports/sentinel1_raw_2017.csv

Existing files preserved:
  harvest_gee_exports/sen